In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re

import sys
sys.path.append('../../..')
# from mount_drive import mount_s_drive

In [2]:
# mount_s_drive(subfolder='LCICM/Databases/eICU')

In [3]:
myHours = 60 * 6

In [4]:
database_folder = '/projects/LCICM/eICU/'

### Patients 

In [5]:
patients_df = pd.read_csv(database_folder+'patient.csv')

In [6]:
patients_df.columns

Index(['patientunitstayid', 'patienthealthsystemstayid', 'gender', 'age',
       'ethnicity', 'hospitalid', 'wardid', 'apacheadmissiondx',
       'admissionheight', 'hospitaladmittime24', 'hospitaladmitoffset',
       'hospitaladmitsource', 'hospitaldischargeyear',
       'hospitaldischargetime24', 'hospitaldischargeoffset',
       'hospitaldischargelocation', 'hospitaldischargestatus', 'unittype',
       'unitadmittime24', 'unitadmitsource', 'unitvisitnumber', 'unitstaytype',
       'admissionweight', 'dischargeweight', 'unitdischargetime24',
       'unitdischargeoffset', 'unitdischargelocation', 'unitdischargestatus',
       'uniquepid'],
      dtype='object')

In [7]:
myIds = pd.read_csv('patient_ids.csv')
myIds

,Unnamed: 0,patientunitstayid
0,93,141816
1,220,142723
2,255,143056
3,601,145204
4,602,145205
...,...,...
4269,200720,3352475
4270,200731,3352512
4271,200778,3352801
4272,200844,3353194


In [8]:
patients_df = patients_df[patients_df['patientunitstayid'].isin(myIds.patientunitstayid)]

In [9]:
# patients_df[patients_df['apacheadmissiondx'].astype(str).str.contains('Cardiac arrest')].patientunitstayid.nunique()

In [10]:
# myIds = patients_df[(patients_df['age'].replace({'> 89': 89}).fillna(0).astype(int) >= 18) & 
#             (patients_df['apacheadmissiondx'].astype(str).str.contains('Cardiac arrest '))].patientunitstayid

In [11]:
myPredictorsDf = patients_df[['patientunitstayid', 'gender', 'age', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitsource', 'admissionweight']]

### Treatments

In [12]:
def getOneHotConditions(aDf, aColumn, aPrefix):
    aDf['conditions'] = aDf[aColumn].str.split('|')
    one_hot = aDf['conditions'].explode().str.get_dummies().groupby(level=0).sum()
    one_hot = one_hot.add_prefix(aPrefix)
    aDf = aDf.drop(columns=['conditions']).join(one_hot)
    return aDf, one_hot

In [13]:
treatment_df = pd.read_csv(database_folder+'treatment.csv')
treatment_df = treatment_df[treatment_df['patientunitstayid'].isin(myIds.patientunitstayid)]

In [14]:
treatment_df_hyp_time = (
    treatment_df[treatment_df['treatmentstring'].astype(str).str.contains('hypothermia')] \
    .groupby('patientunitstayid').agg('first')
)

In [15]:
# treatment_df_hyp_time['treatmentoffset']

patientunitstayid
250724      1256
251469       655
252908       149
252954      2531
256864       975
           ...  
3352203     1970
3352445      451
3352512     5600
3353194       52
3353251    11304
Name: treatmentoffset, Length: 793, dtype: int64

In [16]:
treatment_df, one_hot = getOneHotConditions(treatment_df, 'treatmentstring', 'treatment_')

In [17]:
filtered_df = treatment_df[(treatment_df.treatmentoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df_hyp = treatment_df.groupby('patientunitstayid').agg('sum').reset_index()

In [18]:
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)
filtered_df_hyp['hyp'] = (filtered_df_hyp.treatment_hypothermia != 0).astype(int)
filtered_df_hyp.hyp.sum()

793

In [19]:
treatment_df_hyp_time = treatment_df_hyp_time.reset_index()

In [20]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')
myPredictorsDf1 = myPredictorsDf1.merge(filtered_df_hyp[['patientunitstayid', 'hyp']], on=['patientunitstayid'], how='left')
myPredictorsDf1['treatment_hypothermia'] = myPredictorsDf1.hyp
myPredictorsDf1.drop(columns=['hyp'], inplace = True)

In [21]:
# myPredictorsDf1 = myPredictorsDf1.merge(treatment_df_hyp_time[['patientunitstayid', 'treatmentoffset']], on=['patientunitstayid'], how='left')

In [22]:
# myPredictorsDf1['hypothermia_time'] = myPredictorsDf1['treatmentoffset']

In [23]:
myPredictorsDf = myPredictorsDf1

### Diagnosis

In [24]:
diagnosis_df = pd.read_csv(database_folder+'diagnosis.csv')
diagnosis_df = diagnosis_df[diagnosis_df['patientunitstayid'].isin(myIds.patientunitstayid)]

In [25]:
# myIds = diagnosis_df[diagnosis_df.icd9code.astype(str).str.contains('427.5')].patientunitstayid.unique()

In [26]:
diagnosis_df[diagnosis_df.icd9code.astype(str).str.contains('427.5')]

,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
350,4249756,141816,True,35,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
895,3872672,142723,True,143,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
1033,4171191,143056,False,7,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
1040,3838149,143056,True,900,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
2309,3975271,145204,False,3,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Other
...,...,...,...,...,...,...,...
2710633,46086056,3353251,False,4080,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
2710643,46087802,3353251,False,2653,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Other
2710645,46085958,3353251,False,5521,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary
2710646,46086896,3353251,False,9169,cardiovascular|cardiac arrest|cardiac arrest,"427.5, I46.9",Primary


In [27]:
diagnosis_df, one_hot = getOneHotConditions(diagnosis_df, 'diagnosisstring', 'diagnosis_')

In [28]:
filtered_df = diagnosis_df[(diagnosis_df.diagnosisoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)

In [29]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')

In [30]:
# diagnosis_df['diagnosisstring'].contains('hypothermia'

In [31]:
myPredictorsDf = myPredictorsDf1

### Nurse Charting

In [32]:
file_path = database_folder+'nurseCharting.csv'
chunk_size = 1e6

df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds.patientunitstayid)]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

nurse_charting_df = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

Chunk 0 Processed.
Chunk 10 Processed.
Chunk 20 Processed.
Chunk 30 Processed.
Chunk 40 Processed.
Chunk 50 Processed.
Chunk 60 Processed.
Chunk 70 Processed.
Chunk 80 Processed.
Chunk 90 Processed.
Chunk 100 Processed.
Chunk 110 Processed.
Chunk 120 Processed.
Chunk 130 Processed.
Chunk 140 Processed.
Chunk 150 Processed.
Processing finished.


### Hypothermmia Analysis

In [33]:
myDfTempC = nurse_charting_df[(nurse_charting_df.nursingchartcelltypevalname == 'Temperature (C)')]
myDfTempC.nursingchartvalue = myDfTempC.nursingchartvalue.astype(float)
myDfTempC.sort_values(by=['patientunitstayid', 'nursingchartentryoffset'], inplace=True)
# myDfTempC['TimeDiff'] = myDfTempC['nursingchartoffset'].diff()
# myDfTempC['TempTimesTime'] = myDfTempC['TimeDiff'] * myDfTempC['nursingchartvalue'].shift(1)
# myDfTempC['RollingTemp'] = myDfTempC['TempTimesTime'].rolling(window=2).sum()
# myDfTempC['RollingTime'] = myDfTempC['TimeDiff'].rolling(window=2).sum()
# myDfTempC['AvgTemp'] = myDfTempC['RollingTemp'] / myDfTempC['RollingTime']
# myDfTempC[['TempTimesTime', 'TimeDiff', 'nursingchartvalue', 'RollingTemp', 'RollingTime', 'AvgTemp']]
myDfTempC2 = myDfTempC[(myDfTempC.nursingchartentryoffset < 2880)  \
        & (myDfTempC.nursingchartentryoffset > 0)
        & (myDfTempC.nursingchartvalue < 36)]
myDfTempC2['nursingchartentryoffset2'] = myDfTempC2['nursingchartentryoffset']
myHyp = myDfTempC2.groupby('patientunitstayid').agg({'patientunitstayid':'count', 'nursingchartvalue': 'min', 'nursingchartentryoffset':'min', 'nursingchartentryoffset2':'max'})
myHyp['Hyp'] = (myHyp['patientunitstayid'] > 12).astype(int) \
                & (myHyp['nursingchartvalue'] > 25).astype(int) \
                & (myHyp['nursingchartentryoffset2'] - myHyp['nursingchartentryoffset'] > 720).astype(int)
myHyp[(myHyp['Hyp'] == 1)] 

/tmp/ipykernel_3263073/3276566090.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myDfTempC.nursingchartvalue = myDfTempC.nursingchartvalue.astype(float)
/tmp/ipykernel_3263073/3276566090.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myDfTempC.sort_values(by=['patientunitstayid', 'nursingchartentryoffset'], inplace=True)
/tmp/ipykernel_3263073/3276566090.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

,patientunitstayid,nursingchartvalue,nursingchartentryoffset,nursingchartentryoffset2,Hyp
patientunitstayid,,,,,
151437,28,33.4,15,1246,1
153418,116,33.0,11,2336,1
161900,32,33.1,351,1971,1
165286,104,32.4,468,2778,1
166054,135,33.3,259,2584,1
...,...,...,...,...,...
3352123,115,33.0,12,1872,1
3352203,68,32.6,151,2101,1
3352445,59,33.0,71,2141,1


In [34]:
myPredictorsDf['Hypothermia'] = 0
myPredictorsDf.loc[myPredictorsDf.patientunitstayid.isin(myHyp[myHyp['Hyp'] == 1].index), 'Hypothermia'] = 1
myPredictorsDf[['Hypothermia', 'patientunitstayid']]
myPredictorsDf.Hypothermia.sum()
# myPredictorsDf.drop(columns='Hypothermia', inplace=True)

916

In [35]:
myHyp = myHyp.rename(columns={'patientunitstayid': 'test'}).reset_index()[
                                        ['patientunitstayid', 
                                        'nursingchartentryoffset']]

In [36]:
myHyp

,patientunitstayid,nursingchartentryoffset
0,143056,84
1,145205,999
2,146619,124
3,151437,15
4,153418,11
...,...,...
2213,3352445,71
2214,3352475,1907
2215,3352512,22
2216,3353194,23


In [37]:
myPredictorsDf

,patientunitstayid,gender,age,apacheadmissiondx,admissionheight,hospitaladmittime24,hospitaladmitsource,admissionweight,treatment_10-15 cm H2O,treatment_1500-2000 ml,...,diagnosis_with vomiting,diagnosis_without foley catheter,diagnosis_without hemodynamic compromise,diagnosis_without in-dwelling catheter,diagnosis_without return to pre-existing conscious level,"diagnosis_witnessed, < 15 minutes CPR","diagnosis_witnessed, > 15 minutes CPR",diagnosis_wound dehiscence / evisceration,diagnosis_wound infection,Hypothermia
0,141816,Male,55,Cardiac arrest (with or without respiratory ar...,185.4,20:35:00,Direct Admit,110.70,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,142723,Female,68,Cardiac arrest (with or without respiratory ar...,165.1,21:48:29,Floor,50.30,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,143056,Male,78,Cardiac arrest (with or without respiratory ar...,177.8,17:43:00,Emergency Department,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,145204,Female,23,Cardiac arrest (with or without respiratory ar...,167.6,11:49:00,Emergency Department,64.50,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,145205,Female,23,Cardiac arrest (with or without respiratory ar...,167.6,11:49:00,Emergency Department,64.50,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4269,3352475,Female,63,Cardiac arrest (with or without respiratory ar...,160.0,21:13:00,Floor,58.80,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
4270,3352512,Male,70,Cardiac arrest (with or without respiratory ar...,170.2,23:07:00,Emergency Department,104.30,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4271,3352801,Female,60,Cardiac arrest (with or without respiratory ar...,165.1,05:03:00,Emergency Department,79.40,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4272,3353194,Female,51,Cardiac arrest (with or without respiratory ar...,170.2,07:17:00,Emergency Department,63.05,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [38]:
myPredictorsDf = myPredictorsDf.merge(myHyp, 
                                      on=['patientunitstayid'], how='left')


TypeError: 'set' object is not callable

In [39]:
myPredictorsDf.rename(columns={'nursingchartentryoffset':'hypothermia_time'}, inplace=True)

In [40]:
myPredictorsDf['both_hypothermia'] = ((myPredictorsDf.treatment_hypothermia == 1) | (myPredictorsDf.Hypothermia == 1)).astype(int)

In [41]:
(myDfTempC[(myDfTempC.patientunitstayid ==3243869) & (myDfTempC.nursingchartentryoffset < 500)])

,nursingchartid,patientunitstayid,nursingchartoffset,nursingchartentryoffset,nursingchartcelltypecat,nursingchartcelltypevallabel,nursingchartcelltypevalname,nursingchartvalue
5079504,2241588686,3243869,19,19,Vital Signs,Temperature,Temperature (C),35.0
5079952,2242081401,3243869,34,34,Vital Signs,Temperature,Temperature (C),35.2
5079745,2218814493,3243869,49,49,Vital Signs,Temperature,Temperature (C),35.3
5078415,2230898102,3243869,64,64,Vital Signs,Temperature,Temperature (C),35.4
5078250,2225685817,3243869,79,79,Vital Signs,Temperature,Temperature (C),35.5
5080013,2272143813,3243869,94,94,Vital Signs,Temperature,Temperature (C),35.5
5078255,2210630692,3243869,109,109,Vital Signs,Temperature,Temperature (C),35.4
5079419,2272237371,3243869,124,124,Vital Signs,Temperature,Temperature (C),35.4
5079799,2258855696,3243869,139,139,Vital Signs,Temperature,Temperature (C),35.4
5078359,2260942118,3243869,154,154,Vital Signs,Temperature,Temperature (C),35.4


### Getting Features 

In [42]:
def getFeaturesFromDf(aDf, aTimeColumn, aTypeColumn, aValueColumn):
    aDf['num_values'] = pd.to_numeric(aDf[aValueColumn], errors='coerce')
    aDf = aDf[['patientunitstayid', aTypeColumn, 'num_values', aTimeColumn]].copy()
    myMasterGroupDf = aDf[(aDf[aTimeColumn] < myHours)].groupby(['patientunitstayid', aTypeColumn])
    myMasterGroupBegin = aDf.loc[myMasterGroupDf[aTimeColumn].idxmin()]
    myMasterGroupEnd = aDf.loc[myMasterGroupDf[aTimeColumn].idxmax()]
    myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})
    myMasterGroupAgg = myMasterGroupAgg.num_values.reset_index()
    return myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg

In [43]:
def mergeFeaturesInDf(aDfToMerge, aBegin, aEnd, aAgg, aPrefix, aTypeColumn, aValueColumn):
    print('Running Begin Columns')
    total = int(aBegin[aTypeColumn].nunique() / 10)
    myPredictorsDf1 = aDfToMerge
    i = 0
    print('progress: ', end="")
    for value in aBegin[aTypeColumn].unique():
        myDf = aBegin[aBegin[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running End Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aEnd[aTypeColumn].unique():
        myDf = aEnd[aEnd[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_last_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running Agg Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aAgg[aTypeColumn].unique():
        myDf = aAgg[aAgg[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', 'max', 'min', 'mean']], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
        myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
        myPredictorsDf1[aPrefix + '_mean_' + value] = myPredictorsDf1['mean']
        myPredictorsDf1.drop(columns=['max', 'min', 'mean'], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    return myPredictorsDf1

In [44]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(nurse_charting_df, 'nursingchartoffset', 'nursingchartcelltypevalname', 'nursingchartvalue')

In [45]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'nurse', 'nursingchartcelltypevalname', 'num_values')

Running Begin Columns
progress: #############
Running End Columns
progress: #############
Running Agg Columns
progress: #############

In [46]:
print(list(myPredictorsDf.columns))

['patientunitstayid', 'gender', 'age', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitsource', 'admissionweight', 'treatment_10-15 cm H2O', 'treatment_1500-2000 ml', 'treatment_25-30%', 'treatment_30-40%', 'treatment_40-50%', 'treatment_5-10 cm H2O', 'treatment_50-60%', 'treatment_500-1000 ml', 'treatment_60-70%', 'treatment_70-80%', 'treatment_80-90%', 'treatment_> 15 cm H2O', 'treatment_>250 cc/hr', 'treatment_>90%', 'treatment_ACE inhibitor', 'treatment_AFB', 'treatment_AICD placement', 'treatment_BAL/PBS', 'treatment_C A V H D', 'treatment_C V V H', 'treatment_C V V H D', 'treatment_C. difficile toxin', 'treatment_CABG', 'treatment_CABG and valve', 'treatment_CPAP/PEEP therapy', 'treatment_CSF', 'treatment_CSF drainage via ventriculostomy', 'treatment_CT scan', 'treatment_Cardiac surgery consultation', 'treatment_Cardiology consultation', 'treatment_D10W', "treatment_D5 Lactated Ringer's", 'treatment_D5 half-normal saline', 'treatment_D50', 'treatment_

In [47]:
myPredictorsDf = myPredictorsDf1

In [48]:
gcs_df = nurse_charting_df[nurse_charting_df['nursingchartcelltypevalname']=='GCS Total']

In [49]:
motor_gcs_df = nurse_charting_df[nurse_charting_df.nursingchartcelltypevalname == 'Motor']

In [50]:
myGcsDf = gcs_df.sort_values(by=['patientunitstayid', "nursingchartoffset"])
myGcsDf['nursingchartoffset2'] = myGcsDf.nursingchartoffset
myGcsDf2 = myGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myGcsDfMin = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myGcsDfMax = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [51]:
myMGcsDf = motor_gcs_df.sort_values(by=['patientunitstayid', 'nursingchartoffset'])
myMGcsDf['nursingchartoffset2'] = myMGcsDf.nursingchartoffset
myMGcsDf2 = myMGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myMGcsDfMin = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myMGcsDfMax = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [52]:
myMGcsDfMax

,nursingchartid,patientunitstayid,nursingchartoffset_x,nursingchartentryoffset,nursingchartcelltypecat,nursingchartcelltypevallabel,nursingchartcelltypevalname,nursingchartvalue,num_values,nursingchartoffset2,nursingchartoffset_y
0,138825186,168324,108,216,Scores,Glasgow coma score,Motor,1,1.0,108,108
1,298939097,172465,7992,8023,Scores,Glasgow coma score,Motor,6,6.0,7992,7992
2,265150582,206908,273,280,Scores,Glasgow coma score,Motor,1,1.0,273,273
3,310068282,249143,23,23,Scores,Glasgow coma score,Motor,1,1.0,23,23
4,314703450,249458,1635,1784,Scores,Glasgow coma score,Motor,6,6.0,1635,0
...,...,...,...,...,...,...,...,...,...,...,...
2835,2287942733,3351806,4566,4566,Scores,Glasgow coma score,Motor,1,1.0,4566,3132
2836,2281576142,3352034,11063,11063,Scores,Glasgow coma score,Motor,6,6.0,11063,-15086
2837,2284258361,3352159,62,62,Scores,Glasgow coma score,Motor,1,1.0,62,62
2838,2290230463,3352475,22240,22240,Scores,Glasgow coma score,Motor,6,6.0,22240,-920


In [53]:
myPredictorsDf = myPredictorsDf.merge(myGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstGCSTime', 'nursingchartvalue': 'FirstGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastGCSTime', 'nursingchartvalue': 'LastGCS'}, inplace=True)

In [54]:
myPredictorsDf = myPredictorsDf.merge(myMGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstMGCSTime', 'nursingchartvalue': 'FirstMGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myMGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastMGCSTime', 'nursingchartvalue': 'LastMGCS'}, inplace=True)

### Labs

In [55]:
file_path = database_folder+'lab.csv'
chunk_size = 1e6

# patient_unit_stay_ids = ca_patients_df['patientunitstayid']
df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds.patientunitstayid)]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

myLabs = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

Chunk 0 Processed.
Chunk 10 Processed.
Chunk 20 Processed.
Chunk 30 Processed.
Processing finished.


In [56]:
myLabs

,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,49494540,141816,58,1,LDL,57.000,57,mg/dL,mg/dL,105
1,57313768,141816,348,4,urinary specific gravity,NaN,>1.030,NaN,NaN,418
2,56089485,141816,3783,3,-polys,78.000,78,%,%,3825
3,49828945,141816,2228,3,RBC,3.050,3.05,M/mcL,mil/mcL,2287
4,45664969,141816,788,1,bicarbonate,23.000,23,mmol/L,mmol/L,836
...,...,...,...,...,...,...,...,...,...,...
1314916,826720664,3353251,4049,4,bedside glucose,129.000,129,mg/dL,mg/dL,4049
1314917,822488872,3353251,1849,1,BUN,32.000,32,mg/dL,mg/dL,1894
1314918,829194475,3353251,310,7,pH,7.194,7.194,NaN,NaN,310
1314919,822355171,3353251,409,1,potassium,4.400,4.4,mmol/L,mmol/L,451


In [58]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(myLabs, 
            'labresultoffset', 'labname', 'labresult')

In [59]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, 
                                    myMasterGroupBegin, 
                                    myMasterGroupEnd, 
                                    myMasterGroupAgg, 
                                    'lab', 
                                    'labname', 
                                    'num_values')

Running Begin Columns
progress: ###########
Running End Columns
progress: ###########
Running Agg Columns
progress: ###########

In [60]:
[x for x in myPredictorsDf.columns if 'first' in x]

['treatment_first generation cephalosporin',
 'diagnosis_first degree',
 'nurse_first_GCS Total',
 'nurse_first_Heart Rate',
 'nurse_first_Non-Invasive BP Diastolic',
 'nurse_first_Non-Invasive BP Systolic',
 'nurse_first_O2 Saturation',
 'nurse_first_Pain Score',
 'nurse_first_Respiratory Rate',
 'nurse_first_Temperature (C)',
 'nurse_first_Temperature (F)',
 'nurse_first_Temperature Location',
 'nurse_first_Value',
 'nurse_first_Non-Invasive BP Mean',
 'nurse_first_Invasive BP Diastolic',
 'nurse_first_Invasive BP Systolic',
 'nurse_first_CI',
 'nurse_first_PVR',
 'nurse_first_Bedside Glucose',
 'nurse_first_Eyes',
 'nurse_first_Motor',
 'nurse_first_Verbal',
 'nurse_first_Invasive BP Mean',
 'nurse_first_O2 Admin Device',
 'nurse_first_End Tidal CO2',
 'nurse_first_Delirium Scale',
 'nurse_first_Delirium Score',
 'nurse_first_O2 L/%',
 'nurse_first_Sedation Scale',
 'nurse_first_Sedation Score',
 'nurse_first_CVP',
 'nurse_first_CO',
 'nurse_first_SV',
 'nurse_first_Pain Goal',
 'nu

In [61]:
myPredictorsDf = myPredictorsDf1

In [62]:
myPredictorsDf = myPredictorsDf.merge(patients_df[['patientunitstayid', 'hospitaldischargestatus']], on=['patientunitstayid'])

In [63]:
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf.hospitaldischargestatus == 'Expired'
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf['DeathAtDischarge'].astype(int)

In [64]:
myPredictorsDf['LastGCS15'] = (myPredictorsDf['LastGCS'] == '15').astype(int)

In [65]:
myPredictorsDf.loc[myPredictorsDf.age == '> 89', 'age'] = 89

In [66]:
myPredictorsDf.age = myPredictorsDf.fillna(0).age.astype(int)

In [67]:
myPredictorsDf = myPredictorsDf[myPredictorsDf.age >= 18]

In [68]:
# myPredictorsDf

In [91]:
myPredictorsDf.to_csv('eICUPredictorsDiag.csv', index=False)

In [70]:
# myPredictorsDf[~myPredictorsDf.FirstMGCS.isna()]

In [71]:
# myPredictorsDf.rename(columns={'treatmentoffset': 'hypothermia_time'}, inplace=True)

In [72]:
columns_orig = ['SUBJID',
 'groupe',
 'CPC_SC3',
 'J0_SEX',
 'J0_TAILLE',
 'J0_POIDS',
 'J0_BMI',
 'J0_AGE',
 'J0_PAS',
 'J0_PAD',
 'J0_PAM',
 'J0_FC',
 'J0_SPO2',
 'J0_GLASGOW',
 'J0_GLASGOW_CONTROLE',
 'J0_OCULAIRE',
 'J0_VERBALE',
 'J0_MOTRICE',
 'J0_PUPILG',
 'J0_PUPILG_REA',
 'J0_PUPILD',
 'J0_PUPILD_REA',
 'J0_CILIAIRE',
 'J0_CORNEEN',
 'J0_REFLEXVEST',
 'J0_REFLEXCEPH',
 'J0_REFLEXCARD',
 'J0_TEMP',
 'J0_IGSII',
 'J0_MCCABE',
 'J0_KNAUS',
 'J0_CHARLSON1',
 'J0_CHARLSON2',
 'J0_CHARLSON3',
 'J0_CHARLSON4',
 'J0_CHARLSON5',
 'J0_CHARLSON6',
 'J0_CHARLSON7',
 'J0_CHARLSON8',
 'J0_CHARLSON9',
 'J0_CHARLSON10',
 'J0_CHARLSON11',
 'J0_CHARLSON12',
 'J0_CHARLSON13',
 'J0_CHARLSON14',
 'V0_CHARLSON15',
 'V0_CHARLSON16',
 'V0_CHARLSON17',
 'V0_CHARLSON18',
 'V0_CHARLSON18B',
 'V0_CHARLSON19',
 'J0_CHARLSON',
 'J0_ATCD',
 'J0_CARDIO',
 'J0_NYHA',
 'J0_MYOCARD',
 'J0_ARTERIO',
 'J0_HTA',
 'J0_POUMON',
 'J0_IRC',
 'J0_HYPERCAP',
 'J0_O2',
 'J0_TABAC',
 'J0_RYTHM',
 'J0_CAUSE2_ACR',
 'J0_DSA',
 'J0_DSA_P',
 'J0_TEMOIN',
 'J0_TEMOIN_MASSE',
 'J0_LIEU_ACR',
 'J0_NOFLOW',
 'J0_LOWFLOW',
 'J0_ADRE',
 'J0_ADRE_DOS',
 'J0_CORDA',
 'J0_CORDA_DOS',
 'J0_BICARB',
 'J0_BICARB_DOS',
 'V0_REFROIDI',
 'V0_CHOC_AV',
 'V0_CHOC_AP',
 'V0_PLANCHE',
 'V0_THROMBO',
 'V0_CORO_ACR',
 'V0_ANGIO_ACR',
 'V0_ANGIO_YES',
 'V0_BALLON',
 'V0_ACR2',
 'J0_CURAR',
 'J0_SEDATIF',
 'J0_MORPHIN',
 'J0_COAGUL',
 'J0_AGREG',
 'J0_ANTIBIO',
 'J0_AMINE',
 'J0_NORA',
 'J0_ADRE2',
 'J0_DOBU',
 'J0_DOPA',
 'J0_PEP',
 'J0_FIO2',
 'J0_VT',
 'J0_FR',
 'BIO_LEUCO',
 'BIO_HEMO',
 'BIO_PLAQ',
 'BIO_TP',
 'BIO_DDIMERE',
 'BIO_SODIUM',
 'BIO_POTAS',
 'BIO_UREE',
 'BIO_CREAT',
 'BIO_CALCIUM',
 'BIO_MAGNE',
 'BIO_GLYCEMI',
 'BIO_PROTID',
 'BIO_LIPAS',
 'BIO_TROPO',
 'BIO_TEMP',
 'BIO_FIO2',
 'BIO_PH',
 'BIO_PAO2',
 'BIO_PACO2',
 'BIO_BICARB',
 'BIO_LACTAT',
 'BIO_TROPO2',
 'BIO_TROPO_CGT',
 'ECG',
 'ECG_QTC',
 'ECG_ANOMALI',
 'ECG_SUS_ST',
 'ECG_SOUS_ST',
 'ECG_BAVI',
 'ECG_BAVII',
 'ECG_BAVIII',
 'ECG_BBG',
 'ECG_BBD',
 'ECG_TACHICARD',
 'ECG_FIBRIL',
 'ECG_SALV_VENT',
 'ECG_FLUTER',
 'ECG_SALV_SUPRA',
 'SOFA_SC',
 'SOFA_RESPIR',
 'SOFA_CARDIO',
 'SOFA_COAG',
 'SOFA_NEURO',
 'SOFA_FOIE',
 'SOFA_RENAL',
 'EI_EI',
 'EI_HEMOSEVER',
 'EI_TRANSFUS',
 'EI_INTRACER',
 'EI_CHIR',
 'EI_EXTRARENAL',
 'EI_OAP',
 'EI_ECHO',
 'EI_DIURETIQ',
 'EI_CONVULS',
 'EI_ARYTHMI',
 'EI_ANTIEPILEPTIQ',
 'BARTHEL_SC',
 'SOFA_SC7',
 'SOFA_SC1',
 'DS_DC',
 'DAYS_ALIVE_30',
 'CPC12',
 'SEX']

In [90]:
columns = [
    'patientunitstayid',
    'gender',
    'age',
    'hospitaladmittime24',
    'bmi',
    'nurse_first_Non-Invasive BP Diastolic',
    'nurse_first_Non-Invasive BP Systolic',
    'nurse_first_Non-Invasive BP Mean',
    'nurse_first_Heart Rate',
    'nurse_first_O2 Saturation',
    'nurse_first_Temperature (C)',
    'nurse_first_GCS Total',
    'nurse_last_GCS Total',
    'nurse_max_GCS Total',
    'nurse_min_GCS Total',
    'nurse_mean_GCS Total',
    'FirstGCSTime',
    'FirstGCS',
    'LastGCSTime',
    'LastGCS',
    'FirstMGCSTime',
    'FirstMGCS',
    'LastMGCSTime',
    'LastMGCS',
    'LastGCS15',
    'nurse_first_Motor',
    'diagnosis_initial rhythm: ventricular fibrillation',
    'diagnosis_ventricular fibrillation',
    'diagnosis_ventricular tachycardia',
    'diagnosis_initial rhythm: ventricular tachycardia',
    'diagnosis_initial rhythm: pulseless electrical activity',
    'diagnosis_initial rhythm: asystole',
    'hypothermia_time',
    'lab_first_Carboxyhemoglobin',
    'lab_first_Methemoglobin',
    'lab_first_Oxyhemoglobin',
    'lab_last_Carboxyhemoglobin',
    'lab_last_Methemoglobin',
    'lab_last_Oxyhemoglobin',
    'lab_max_Carboxyhemoglobin',
    'lab_min_Carboxyhemoglobin',
    'lab_mean_Carboxyhemoglobin',
    'lab_max_Methemoglobin',
    'lab_min_Methemoglobin',
    'lab_mean_Methemoglobin',
    'lab_max_Oxyhemoglobin',
    'lab_min_Oxyhemoglobin',
    'lab_mean_Oxyhemoglobin',
    'nurse_first_QTc',
    'nurse_last_QTc',
    'nurse_max_QTc',
    'nurse_min_QTc',
    'nurse_mean_QTc',
    'lab_first_-basos',
 'lab_first_-eos',
 'lab_first_-lymphs',
 'lab_first_-monos',
 'lab_first_-polys',
 'lab_first_BUN',
 'lab_first_HDL',
 'lab_first_Hct',
 'lab_first_Hgb',
 'lab_first_LDL',
 'lab_first_MCH',
 'lab_first_MCHC',
 'lab_first_MCV',
 'lab_first_RBC',
 'lab_first_RDW',
 'lab_first_WBC x 1000',
 'lab_first_anion gap',
 'lab_first_bedside glucose',
 'lab_first_bicarbonate',
 'lab_first_calcium',
 'lab_first_chloride',
 'lab_first_creatinine',
 'lab_first_glucose',
 'lab_first_magnesium',
 'lab_first_platelets x 1000',
 'lab_first_potassium',
 'lab_first_sodium',
 'lab_first_total cholesterol',
 'lab_first_triglycerides',
 'lab_first_troponin - I',
 'lab_first_urinary specific gravity',
 'lab_first_ALT (SGPT)',
 'lab_first_AST (SGOT)',
 'lab_first_Base Deficit',
 'lab_first_FiO2',
 'lab_first_HCO3',
 'lab_first_albumin',
 'lab_first_alkaline phos.',
 'lab_first_lactate',
 'lab_first_pH',
 'lab_first_paCO2',
 'lab_first_paO2',
 'lab_first_total bilirubin',
 'lab_first_total protein',
 'lab_first_PT',
 'lab_first_PT - INR',
 'lab_first_Acetaminophen',
 'lab_first_LPM O2',
 'lab_first_O2 Sat (%)',
 'lab_first_Total CO2',
 'lab_first_direct bilirubin',
 'lab_first_ethanol',
 'lab_first_salicylate',
 'lab_first_Base Excess',
 'lab_first_CPK',
 'lab_first_Fe',
 'lab_first_Fe/TIBC Ratio',
 'lab_first_Ferritin',
 'lab_first_PTT',
 'lab_first_TIBC',
 'lab_first_Vancomycin - trough',
 'lab_first_phosphate',
 'lab_first_uric acid',
 'lab_first_urinary creatinine',
 'lab_first_urinary sodium',
 'lab_first_Carboxyhemoglobin',
 'lab_first_Methemoglobin',
 'lab_first_Vitamin B12',
 'lab_first_folate',
 'lab_first_ionized calcium',
 'lab_first_-bands',
 'lab_first_BNP',
 'lab_first_ammonia',
 'lab_first_cortisol',
 'lab_first_free T4',
 'lab_first_lipase',
 'lab_first_CRP',
 'lab_first_fibrinogen',
 'lab_first_TSH',
 'lab_first_CPK-MB',
 'lab_first_myoglobin',
 'lab_first_Phenytoin',
 'lab_first_LDH',
 "lab_first_WBC's in urine",
 'lab_first_amylase',
 'lab_first_Respiratory Rate',
 'lab_first_Digoxin',
 'lab_first_Vancomycin - random',
 'lab_first_prolactin',
 'lab_first_serum osmolality',
 'lab_first_T3',
 'lab_first_haptoglobin',
 'lab_first_reticulocyte count',
 'lab_first_ESR',
 'lab_first_urinary osmolality',
 'lab_first_PEEP',
 'lab_first_TV',
 'lab_first_Temperature',
 'lab_first_Vent Rate',
 'lab_first_troponin - T',
 'lab_first_CPK-MB INDEX',
 'lab_first_MPV',
 'lab_first_PTT ratio',
 'lab_first_Pressure Support',
 'lab_first_prealbumin',
 'lab_first_Oxyhemoglobin',
 'lab_first_Peak Airway/Pressure',
 'lab_first_Lithium',
 'lab_first_O2 Content',
 'lab_first_24 h urine protein',
 "lab_first_WBC's in pleural fluid",
 'lab_first_T3RU',
 'lab_first_T4',
 "lab_first_WBC's in cerebrospinal fluid",
 'lab_first_glucose - CSF',
 'lab_first_protein - CSF',
 'lab_first_transferrin',
 'lab_first_Spontaneous Rate',
 'lab_first_Tobramycin - random',
 'lab_first_Tacrolimus-FK506',
 'lab_first_Device',
 'lab_first_Mode',
 'lab_first_Carbamazepine',
 'lab_first_serum ketones',
 'lab_first_Pressure Control',
 'lab_first_Vent Other',
 'lab_first_Theophylline',
 'lab_first_Phenobarbital',
 "lab_first_WBC's in synovial fluid",
 'lab_first_protein C',
 'lab_first_protein S',
 'lab_first_Cyclosporin',
 'lab_first_Lidocaine',
 'lab_first_CRP-hs',
 "lab_first_WBC's in body fluid",
 'lab_first_Gentamicin - random',
 'lab_first_24 h urine urea nitrogen',
 'lab_first_ANF/ANA',
 'lab_first_Site',
 'lab_first_cd 4',
 'lab_first_Tobramycin - peak',
 'lab_first_Tobramycin - trough',
 'lab_first_Clostridium difficile toxin A+B',
 'lab_last_-basos',
 'lab_last_-eos',
 'lab_last_-lymphs',
 'lab_last_-monos',
 'lab_last_-polys',
 'lab_last_BUN',
 'lab_last_HDL',
 'lab_last_Hct',
 'lab_last_Hgb',
 'lab_last_LDL',
 'lab_last_MCH',
 'lab_last_MCHC',
 'lab_last_MCV',
 'lab_last_RBC',
 'lab_last_RDW',
 'lab_last_WBC x 1000',
 'lab_last_anion gap',
 'lab_last_bedside glucose',
 'lab_last_bicarbonate',
 'lab_last_calcium',
 'lab_last_chloride',
 'lab_last_creatinine',
 'lab_last_glucose',
 'lab_last_magnesium',
 'lab_last_platelets x 1000',
 'lab_last_potassium',
 'lab_last_sodium',
 'lab_last_total cholesterol',
 'lab_last_triglycerides',
 'lab_last_troponin - I',
 'lab_last_urinary specific gravity',
 'lab_last_ALT (SGPT)',
 'lab_last_AST (SGOT)',
 'lab_last_Base Deficit',
 'lab_last_FiO2',
 'lab_last_HCO3',
 'lab_last_albumin',
 'lab_last_alkaline phos.',
 'lab_last_lactate',
 'lab_last_pH',
 'lab_last_paCO2',
 'lab_last_paO2',
 'lab_last_total bilirubin',
 'lab_last_total protein',
 'lab_last_PT',
 'lab_last_PT - INR',
 'lab_last_Acetaminophen',
 'lab_last_LPM O2',
 'lab_last_O2 Sat (%)',
 'lab_last_Total CO2',
 'lab_last_direct bilirubin',
 'lab_last_ethanol',
 'lab_last_salicylate',
 'lab_last_Base Excess',
 'lab_last_CPK',
 'lab_last_Fe',
 'lab_last_Fe/TIBC Ratio',
 'lab_last_Ferritin',
 'lab_last_PTT',
 'lab_last_TIBC',
 'lab_last_Vancomycin - trough',
 'lab_last_phosphate',
 'lab_last_uric acid',
 'lab_last_urinary creatinine',
 'lab_last_urinary sodium',
 'lab_last_Carboxyhemoglobin',
 'lab_last_Methemoglobin',
 'lab_last_Vitamin B12',
 'lab_last_folate',
 'lab_last_ionized calcium',
 'lab_last_-bands',
 'lab_last_BNP',
 'lab_last_ammonia',
 'lab_last_cortisol',
 'lab_last_free T4',
 'lab_last_lipase',
 'lab_last_CRP',
 'lab_last_fibrinogen',
 'lab_last_TSH',
 'lab_last_CPK-MB',
 'lab_last_myoglobin',
 'lab_last_Phenytoin',
 'lab_last_LDH',
 "lab_last_WBC's in urine",
 'lab_last_amylase',
 'lab_last_Respiratory Rate',
 'lab_last_Digoxin',
 'lab_last_Vancomycin - random',
 'lab_last_prolactin',
 'lab_last_serum osmolality',
 'lab_last_T3',
 'lab_last_haptoglobin',
 'lab_last_reticulocyte count',
 'lab_last_ESR',
 'lab_last_urinary osmolality',
 'lab_last_PEEP',
 'lab_last_TV',
 'lab_last_Temperature',
 'lab_last_Vent Rate',
 'lab_last_troponin - T',
 'lab_last_CPK-MB INDEX',
 'lab_last_MPV',
 'lab_last_PTT ratio',
 'lab_last_Pressure Support',
 'lab_last_prealbumin',
 'lab_last_Oxyhemoglobin',
 'lab_last_Peak Airway/Pressure',
 'lab_last_Lithium',
 'lab_last_O2 Content',
 'lab_last_24 h urine protein',
 "lab_last_WBC's in pleural fluid",
 'lab_last_T3RU',
 'lab_last_T4',
 "lab_last_WBC's in cerebrospinal fluid",
 'lab_last_glucose - CSF',
 'lab_last_protein - CSF',
 'lab_last_transferrin',
 'lab_last_Spontaneous Rate',
 'lab_last_Tobramycin - random',
 'lab_last_Tacrolimus-FK506',
 'lab_last_Device',
 'lab_last_Mode',
 'lab_last_Carbamazepine',
 'lab_last_serum ketones',
 'lab_last_Pressure Control',
 'lab_last_Vent Other',
 'lab_last_Theophylline',
 'lab_last_Phenobarbital',
 "lab_last_WBC's in synovial fluid",
 'lab_last_protein C',
 'lab_last_protein S',
 'lab_last_Cyclosporin',
 'lab_last_Lidocaine',
 'lab_last_CRP-hs',
 "lab_last_WBC's in body fluid",
 'lab_last_Gentamicin - random',
 'lab_last_24 h urine urea nitrogen',
 'lab_last_ANF/ANA',
 'lab_last_Site',
 'lab_last_cd 4',
 'lab_last_Tobramycin - peak',
 'lab_last_Tobramycin - trough',
 'lab_last_Clostridium difficile toxin A+B',
 'lab_max_-basos',
 'lab_min_-basos',
 'lab_mean_-basos',
 'lab_max_-eos',
 'lab_min_-eos',
 'lab_mean_-eos',
 'lab_max_-lymphs',
 'lab_min_-lymphs',
 'lab_mean_-lymphs',
 'lab_max_-monos',
 'lab_min_-monos',
 'lab_mean_-monos',
 'lab_max_-polys',
 'lab_min_-polys',
 'lab_mean_-polys',
 'lab_max_BUN',
 'lab_min_BUN',
 'lab_mean_BUN',
 'lab_max_HDL',
 'lab_min_HDL',
 'lab_mean_HDL',
 'lab_max_Hct',
 'lab_min_Hct',
 'lab_mean_Hct',
 'lab_max_Hgb',
 'lab_min_Hgb',
 'lab_mean_Hgb',
 'lab_max_LDL',
 'lab_min_LDL',
 'lab_mean_LDL',
 'lab_max_MCH',
 'lab_min_MCH',
 'lab_mean_MCH',
 'lab_max_MCHC',
 'lab_min_MCHC',
 'lab_mean_MCHC',
 'lab_max_MCV',
 'lab_min_MCV',
 'lab_mean_MCV',
 'lab_max_RBC',
 'lab_min_RBC',
 'lab_mean_RBC',
 'lab_max_RDW',
 'lab_min_RDW',
 'lab_mean_RDW',
 'lab_max_WBC x 1000',
 'lab_min_WBC x 1000',
 'lab_mean_WBC x 1000',
 'lab_max_anion gap',
 'lab_min_anion gap',
 'lab_mean_anion gap',
 'lab_max_bedside glucose',
 'lab_min_bedside glucose',
 'lab_mean_bedside glucose',
 'lab_max_bicarbonate',
 'lab_min_bicarbonate',
 'lab_mean_bicarbonate',
 'lab_max_calcium',
 'lab_min_calcium',
 'lab_mean_calcium',
 'lab_max_chloride',
 'lab_min_chloride',
 'lab_mean_chloride',
 'lab_max_creatinine',
 'lab_min_creatinine',
 'lab_mean_creatinine',
 'lab_max_glucose',
 'lab_min_glucose',
 'lab_mean_glucose',
 'lab_max_magnesium',
 'lab_min_magnesium',
 'lab_mean_magnesium',
 'lab_max_platelets x 1000',
 'lab_min_platelets x 1000',
 'lab_mean_platelets x 1000',
 'lab_max_potassium',
 'lab_min_potassium',
 'lab_mean_potassium',
 'lab_max_sodium',
 'lab_min_sodium',
 'lab_mean_sodium',
 'lab_max_total cholesterol',
 'lab_min_total cholesterol',
 'lab_mean_total cholesterol',
 'lab_max_triglycerides',
 'lab_min_triglycerides',
 'lab_mean_triglycerides',
 'lab_max_troponin - I',
 'lab_min_troponin - I',
 'lab_mean_troponin - I',
 'lab_max_urinary specific gravity',
 'lab_min_urinary specific gravity',
 'lab_mean_urinary specific gravity',
 'lab_max_ALT (SGPT)',
 'lab_min_ALT (SGPT)',
 'lab_mean_ALT (SGPT)',
 'lab_max_AST (SGOT)',
 'lab_min_AST (SGOT)',
 'lab_mean_AST (SGOT)',
 'lab_max_Base Deficit',
 'lab_min_Base Deficit',
 'lab_mean_Base Deficit',
 'lab_max_FiO2',
 'lab_min_FiO2',
 'lab_mean_FiO2',
 'lab_max_HCO3',
 'lab_min_HCO3',
 'lab_mean_HCO3',
 'lab_max_albumin',
 'lab_min_albumin',
 'lab_mean_albumin',
 'lab_max_alkaline phos.',
 'lab_min_alkaline phos.',
 'lab_mean_alkaline phos.',
 'lab_max_lactate',
 'lab_min_lactate',
 'lab_mean_lactate',
 'lab_max_pH',
 'lab_min_pH',
 'lab_mean_pH',
 'lab_max_paCO2',
 'lab_min_paCO2',
 'lab_mean_paCO2',
 'lab_max_paO2',
 'lab_min_paO2',
 'lab_mean_paO2',
 'lab_max_total bilirubin',
 'lab_min_total bilirubin',
 'lab_mean_total bilirubin',
 'lab_max_total protein',
 'lab_min_total protein',
 'lab_mean_total protein',
 'lab_max_PT',
 'lab_min_PT',
 'lab_mean_PT',
 'lab_max_PT - INR',
 'lab_min_PT - INR',
 'lab_mean_PT - INR',
 'lab_max_Acetaminophen',
 'lab_min_Acetaminophen',
 'lab_mean_Acetaminophen',
 'lab_max_LPM O2',
 'lab_min_LPM O2',
 'lab_mean_LPM O2',
 'lab_max_O2 Sat (%)',
 'lab_min_O2 Sat (%)',
 'lab_mean_O2 Sat (%)',
 'lab_max_Total CO2',
 'lab_min_Total CO2',
 'lab_mean_Total CO2',
 'lab_max_direct bilirubin',
 'lab_min_direct bilirubin',
 'lab_mean_direct bilirubin',
 'lab_max_ethanol',
 'lab_min_ethanol',
 'lab_mean_ethanol',
 'lab_max_salicylate',
 'lab_min_salicylate',
 'lab_mean_salicylate',
 'lab_max_Base Excess',
 'lab_min_Base Excess',
 'lab_mean_Base Excess',
 'lab_max_CPK',
 'lab_min_CPK',
 'lab_mean_CPK',
 'lab_max_Fe',
 'lab_min_Fe',
 'lab_mean_Fe',
 'lab_max_Fe/TIBC Ratio',
 'lab_min_Fe/TIBC Ratio',
 'lab_mean_Fe/TIBC Ratio',
 'lab_max_Ferritin',
 'lab_min_Ferritin',
 'lab_mean_Ferritin',
 'lab_max_PTT',
 'lab_min_PTT',
 'lab_mean_PTT',
 'lab_max_TIBC',
 'lab_min_TIBC',
 'lab_mean_TIBC',
 'lab_max_Vancomycin - trough',
 'lab_min_Vancomycin - trough',
 'lab_mean_Vancomycin - trough',
 'lab_max_phosphate',
 'lab_min_phosphate',
 'lab_mean_phosphate',
 'lab_max_uric acid',
 'lab_min_uric acid',
 'lab_mean_uric acid',
 'lab_max_urinary creatinine',
 'lab_min_urinary creatinine',
 'lab_mean_urinary creatinine',
 'lab_max_urinary sodium',
 'lab_min_urinary sodium',
 'lab_mean_urinary sodium',
 'lab_max_Carboxyhemoglobin',
 'lab_min_Carboxyhemoglobin',
 'lab_mean_Carboxyhemoglobin',
 'lab_max_Methemoglobin',
 'lab_min_Methemoglobin',
 'lab_mean_Methemoglobin',
 'lab_max_Vitamin B12',
 'lab_min_Vitamin B12',
 'lab_mean_Vitamin B12',
 'lab_max_folate',
 'lab_min_folate',
 'lab_mean_folate',
 'lab_max_ionized calcium',
 'lab_min_ionized calcium',
 'lab_mean_ionized calcium',
 'lab_max_-bands',
 'lab_min_-bands',
 'lab_mean_-bands',
 'lab_max_BNP',
 'lab_min_BNP',
 'lab_mean_BNP',
 'lab_max_ammonia',
 'lab_min_ammonia',
 'lab_mean_ammonia',
 'lab_max_cortisol',
 'lab_min_cortisol',
 'lab_mean_cortisol',
 'lab_max_free T4',
 'lab_min_free T4',
 'lab_mean_free T4',
 'lab_max_lipase',
 'lab_min_lipase',
 'lab_mean_lipase',
 'lab_max_CRP',
 'lab_min_CRP',
 'lab_mean_CRP',
 'lab_max_fibrinogen',
 'lab_min_fibrinogen',
 'lab_mean_fibrinogen',
 'lab_max_TSH',
 'lab_min_TSH',
 'lab_mean_TSH',
 'lab_max_CPK-MB',
 'lab_min_CPK-MB',
 'lab_mean_CPK-MB',
 'lab_max_myoglobin',
 'lab_min_myoglobin',
 'lab_mean_myoglobin',
 'lab_max_Phenytoin',
 'lab_min_Phenytoin',
 'lab_mean_Phenytoin',
 'lab_max_LDH',
 'lab_min_LDH',
 'lab_mean_LDH',
 "lab_max_WBC's in urine",
 "lab_min_WBC's in urine",
 "lab_mean_WBC's in urine",
 'lab_max_amylase',
 'lab_min_amylase',
 'lab_mean_amylase',
 'lab_max_Respiratory Rate',
 'lab_min_Respiratory Rate',
 'lab_mean_Respiratory Rate',
 'lab_max_Digoxin',
 'lab_min_Digoxin',
 'lab_mean_Digoxin',
 'lab_max_Vancomycin - random',
 'lab_min_Vancomycin - random',
 'lab_mean_Vancomycin - random',
 'lab_max_prolactin',
 'lab_min_prolactin',
 'lab_mean_prolactin',
 'lab_max_serum osmolality',
 'lab_min_serum osmolality',
 'lab_mean_serum osmolality',
 'lab_max_T3',
 'lab_min_T3',
 'lab_mean_T3',
 'lab_max_haptoglobin',
 'lab_min_haptoglobin',
 'lab_mean_haptoglobin',
 'lab_max_reticulocyte count',
 'lab_min_reticulocyte count',
 'lab_mean_reticulocyte count',
 'lab_max_ESR',
 'lab_min_ESR',
 'lab_mean_ESR',
 'lab_max_urinary osmolality',
 'lab_min_urinary osmolality',
 'lab_mean_urinary osmolality',
 'lab_max_PEEP',
 'lab_min_PEEP',
 'lab_mean_PEEP',
 'lab_max_TV',
 'lab_min_TV',
 'lab_mean_TV',
 'lab_max_Temperature',
 'lab_min_Temperature',
 'lab_mean_Temperature',
 'lab_max_Vent Rate',
 'lab_min_Vent Rate',
 'lab_mean_Vent Rate',
 'lab_max_troponin - T',
 'lab_min_troponin - T',
 'lab_mean_troponin - T',
 'lab_max_CPK-MB INDEX',
 'lab_min_CPK-MB INDEX',
 'lab_mean_CPK-MB INDEX',
 'lab_max_MPV',
 'lab_min_MPV',
 'lab_mean_MPV',
 'lab_max_PTT ratio',
 'lab_min_PTT ratio',
 'lab_mean_PTT ratio',
 'lab_max_Pressure Support',
 'lab_min_Pressure Support',
 'lab_mean_Pressure Support',
 'lab_max_prealbumin',
 'lab_min_prealbumin',
 'lab_mean_prealbumin',
 'lab_max_Oxyhemoglobin',
 'lab_min_Oxyhemoglobin',
 'lab_mean_Oxyhemoglobin',
 'lab_max_Peak Airway/Pressure',
 'lab_min_Peak Airway/Pressure',
 'lab_mean_Peak Airway/Pressure',
 'lab_max_Lithium',
 'lab_min_Lithium',
 'lab_mean_Lithium',
 'lab_max_O2 Content',
 'lab_min_O2 Content',
 'lab_mean_O2 Content',
 'lab_max_24 h urine protein',
 'lab_min_24 h urine protein',
 'lab_mean_24 h urine protein',
 "lab_max_WBC's in pleural fluid",
 "lab_min_WBC's in pleural fluid",
 "lab_mean_WBC's in pleural fluid",
 'lab_max_T3RU',
 'lab_min_T3RU',
 'lab_mean_T3RU',
 'lab_max_T4',
 'lab_min_T4',
 'lab_mean_T4',
 "lab_max_WBC's in cerebrospinal fluid",
 "lab_min_WBC's in cerebrospinal fluid",
 "lab_mean_WBC's in cerebrospinal fluid",
 'lab_max_glucose - CSF',
 'lab_min_glucose - CSF',
 'lab_mean_glucose - CSF',
 'lab_max_protein - CSF',
 'lab_min_protein - CSF',
 'lab_mean_protein - CSF',
 'lab_max_transferrin',
 'lab_min_transferrin',
 'lab_mean_transferrin',
 'lab_max_Spontaneous Rate',
 'lab_min_Spontaneous Rate',
 'lab_mean_Spontaneous Rate',
 'lab_max_Tobramycin - random',
 'lab_min_Tobramycin - random',
 'lab_mean_Tobramycin - random',
 'lab_max_Tacrolimus-FK506',
 'lab_min_Tacrolimus-FK506',
 'lab_mean_Tacrolimus-FK506',
 'lab_max_Device',
 'lab_min_Device',
 'lab_mean_Device',
 'lab_max_Mode',
 'lab_min_Mode',
 'lab_mean_Mode',
 'lab_max_Carbamazepine',
 'lab_min_Carbamazepine',
 'lab_mean_Carbamazepine',
 'lab_max_serum ketones',
 'lab_min_serum ketones',
 'lab_mean_serum ketones',
 'lab_max_Pressure Control',
 'lab_min_Pressure Control',
 'lab_mean_Pressure Control',
 'lab_max_Vent Other',
 'lab_min_Vent Other',
 'lab_mean_Vent Other',
 'lab_max_Theophylline',
 'lab_min_Theophylline',
 'lab_mean_Theophylline',
 'lab_max_Phenobarbital',
 'lab_min_Phenobarbital',
 'lab_mean_Phenobarbital',
 "lab_max_WBC's in synovial fluid",
 "lab_min_WBC's in synovial fluid",
 "lab_mean_WBC's in synovial fluid",
 'lab_max_protein C',
 'lab_min_protein C',
 'lab_mean_protein C',
 'lab_max_protein S',
 'lab_min_protein S',
 'lab_mean_protein S',
 'lab_max_Cyclosporin',
 'lab_min_Cyclosporin',
 'lab_mean_Cyclosporin',
 'lab_max_Lidocaine',
 'lab_min_Lidocaine',
 'lab_mean_Lidocaine',
 'lab_max_CRP-hs',
 'lab_min_CRP-hs',
 'lab_mean_CRP-hs',
 "lab_max_WBC's in body fluid",
 "lab_min_WBC's in body fluid",
 "lab_mean_WBC's in body fluid",
 'lab_max_Gentamicin - random',
 'lab_min_Gentamicin - random',
 'lab_mean_Gentamicin - random',
 'lab_max_24 h urine urea nitrogen',
 'lab_min_24 h urine urea nitrogen',
 'lab_mean_24 h urine urea nitrogen',
 'lab_max_ANF/ANA',
 'lab_min_ANF/ANA',
 'lab_mean_ANF/ANA',
 'lab_max_Site',
 'lab_min_Site',
 'lab_mean_Site',
 'lab_max_cd 4',
 'lab_min_cd 4',
 'lab_mean_cd 4',
 'lab_max_Tobramycin - peak',
 'lab_min_Tobramycin - peak',
 'lab_mean_Tobramycin - peak',
 'lab_max_Tobramycin - trough',
 'lab_min_Tobramycin - trough',
 'lab_mean_Tobramycin - trough',
 'lab_max_Clostridium difficile toxin A+B',
 'lab_min_Clostridium difficile toxin A+B',
 'lab_mean_Clostridium difficile toxin A+B'
]

In [89]:
# [x for x in myPredictorsDf.columns if 'ele' in x.lower()]

['treatment_administration of electrolytes',
 'treatment_antiplatelet agent',
 'treatment_antiplatelet agents',
 'treatment_citalopram (Celexa)',
 'treatment_coagulation and platelets',
 'treatment_electrical',
 'treatment_electrolyte administration',
 'treatment_electrolyte correction',
 'treatment_intravenous fluids / electrolytes',
 'treatment_platelet concentrate',
 'treatment_tunneled catheter',
 'diagnosis_acute myocardial infarction (no ST elevation)',
 'diagnosis_acute myocardial infarction (with ST elevation)',
 'diagnosis_atelectasis/collapse',
 'diagnosis_due to atelectasis',
 'diagnosis_electrolyte imbalance',
 'diagnosis_fluids and electrolytes',
 'diagnosis_initial rhythm: pulseless electrical activity',
 'diagnosis_platelet disorders',
 'diagnosis_platelet dysfunction',
 'diagnosis_skin, muscle and skeletal tumors',
 'diagnosis_transaminase elevation',
 'diagnosis_trauma - skeletal',
 'nurse_first_Electrolyte Replacement',
 'nurse_last_Electrolyte Replacement',
 'nurse_m

In [75]:
myPredictorsDf[myPredictorsDf.Hypothermia == 1].hypothermia_time

10       11.0
18      351.0
21      468.0
22      259.0
23       12.0
        ...  
3275     12.0
3277    151.0
3279     71.0
3281     22.0
3284     73.0
Name: hypothermia_time, Length: 752, dtype: float64

In [76]:
myPredictorsDf['bmi'] = myPredictorsDf.admissionweight / (myPredictorsDf.admissionheight/100)**2

In [77]:
myPredictorsDf.bmi

0       32.205360
1       18.453291
2       22.962104
3       22.962104
4       44.969557
          ...    
3280    22.968750
3281    36.005197
3282    29.129052
3283    21.765366
3284    32.265371
Name: bmi, Length: 3172, dtype: float64